# Imports

In [1]:
import torch
import sys
import os
from torch import Tensor
sys.path.append('..')
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from definitions import DATA_DIR, EXPERIMENTS_DIR, ROOT_DIR
from src.commands.train import Trainer, Config, ArticleAsKG
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
)
from PIL import ImageColor

In [2]:
px.defaults.template = 'seaborn'

In [3]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Load the results

In [4]:
# TASK = 'hierarchical_classification'
EXPERIMENT = 'default'
CORPUS = 'nantes-uni-2020'
EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, 'results')
HISTORY_PATH = os.path.join(EXPERIMENT_DIR, 'history.csv')
CONFIG_PATH = os.path.join(EXPERIMENT_DIR, 'config.json')
SPLIT_PATH = os.path.join(EXPERIMENT_DIR, 'split_sub_dsis_4.pt')

In [5]:
history = pd.read_csv(HISTORY_PATH)
history = history \
    .drop(columns = ['iteration']) \
    .groupby(['epoch','phase','exp','trial']) \
    .mean() \
    .reset_index()
history.rename(columns = { "exp" : "expr" }, inplace = True)
history.head()

,epoch,phase,expr,trial,guidance_loss,loss,guidance_acc,guidance_f1
0,0,train,Attn_Attn_null_False,0,0.188787,0.188787,0.927726,0.283555
1,0,train,Attn_Attn_null_False,1,0.184929,0.184929,0.927755,0.279122
2,0,train,Attn_Attn_null_False,2,0.189807,0.189807,0.925983,0.269763
3,0,train,Attn_Attn_null_False,3,0.188652,0.188652,0.928155,0.280207
4,0,train,Attn_Attn_null_True,0,0.346495,0.346495,0.846000,0.849118


In [6]:
history_mean = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','expr']).mean() \
    .add_suffix('_mean')

history_std = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','expr']).std() \
    .add_suffix('_std')

In [7]:
history = history_mean.join(other = history_std).reset_index()
history.head()

,epoch,phase,expr,guidance_loss_mean,loss_mean,guidance_acc_mean,guidance_f1_mean,guidance_loss_std,loss_std,guidance_acc_std,guidance_f1_std
0,0,train,Attn_Attn_null_False,0.188044,0.188044,0.927405,0.278161,0.002140,0.002140,0.000968,0.005909
1,0,train,Attn_Attn_null_True,0.339944,0.339944,0.848582,0.850046,0.004424,0.004424,0.002560,0.001683
2,0,val,Attn_Attn_null_False,0.142020,0.142020,0.944205,0.368670,0.003100,0.003100,0.001519,0.014185
3,0,val,Attn_Attn_null_True,0.295292,0.295292,0.870400,0.402034,0.011219,0.011219,0.006384,0.005090
4,1,train,Attn_Attn_null_False,0.124510,0.124510,0.951075,0.422738,0.000749,0.000749,0.000624,0.006289


# Load Data

In [8]:
config = Config(experiment='default')
trainer = Trainer(experiment=config.experiment, config=config.training_config)
configs = trainer.get_configs()

In [9]:
data = trainer.load_data()
data = trainer.filter_data(data)
data = trainer.keep_largest_cc(data) if trainer.config.keep_largest_cc else data
data.split = trainer.create_or_load_split(data)
data.articles.split = data.split

In [10]:
metadata = torch.load(os.path.join(DATA_DIR, CORPUS, 'graph.pt'), weights_only=False)['metadata']

# Learning Curves

In [11]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> None:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['expr'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and expr == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

In [12]:
line_plot(history, 'loss')

In [13]:
line_plot(history, 'guidance_acc')

In [14]:
line_plot(history, 'guidance_f1')

# Calculate other metrics

## Load Predictions

In [15]:
def load_results(results_dir : str) -> dict[tuple[str, str], Tensor]:

    results = dict()

    for filename in os.listdir(results_dir):

        parts = filename.split('.')[0].split('_')
        trial = parts[-1]
        expr = '_'.join(parts[1:-1])

        results[(expr,trial)] = torch.load(os.path.join(RESULTS_DIR, filename), map_location='cpu', weights_only=False)

    return results

In [16]:
results = load_results(RESULTS_DIR)

## Label Wise & Row Wise Metrics

In [17]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [19]:
def get_experiment_params(expr : str) -> dict:

    parts = expr.split('_')
        
    return {
        "GNN Variant" : parts[0],
        "Readout Function" : parts[1],
        "Pooling Layer" : parts[2],
        "Undersampling" : parts[3]
    }

In [20]:
def get_label_wise_metrics(
    data : ArticleAsKG,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    for label in range(data.y.size(-1)):
            
        for (expr,trial),result in results.items():

            y = data.y[data.split == int(trial)]
            y_hat = result[data.split == int(trial)].sigmoid()
            y_hard = y_hat.gt(0.5).long()
            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label]),
                "precision" : precision_score(y[:,label], y_hard[:,label]),
                "recall" : recall_score(y[:,label], y_hard[:,label]),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label])
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [21]:
def get_row_wise_metrics(
    data : ArticleAsKG,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        y = data.y[data.split == int(trial)]
        y_hat = result[data.split == int(trial)].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro'),
            "precision" : precision_score(y, y_hard , average='macro'),
            "recall" : recall_score(y, y_hard , average='macro'),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [22]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics(data.articles, results, metadata['labels'])
display_mean_std_df(label_wise_metrics_agg)

,expr,precision,f1_score,auc,accuracy,recall
0,Attn_Attn_null_False,78.22% ± 1.61%,70.91% ± 0.86%,96.30% ± 0.27%,95.48% ± 0.16%,65.74% ± 1.75%
1,Attn_Attn_null_True,65.86% ± 1.61%,70.55% ± 0.98%,96.67% ± 0.26%,94.39% ± 0.07%,77.02% ± 2.12%
2,Attn_Max_null_False,95.82% ± 0.19%,89.96% ± 0.15%,99.26% ± 0.17%,98.63% ± 0.10%,85.84% ± 0.44%
3,Attn_Max_null_True,78.19% ± 1.92%,84.24% ± 1.68%,98.87% ± 0.11%,96.67% ± 0.44%,92.05% ± 1.29%
4,Attn_Mean_null_False,96.51% ± 0.85%,86.24% ± 1.71%,99.26% ± 0.20%,98.22% ± 0.13%,79.26% ± 2.14%
5,Attn_Mean_null_True,76.51% ± 2.72%,82.77% ± 1.59%,98.88% ± 0.12%,96.50% ± 0.08%,91.10% ± 0.74%
6,Conv_Attn_null_False,94.22% ± 0.88%,91.68% ± 1.37%,99.29% ± 0.16%,98.67% ± 0.16%,89.59% ± 1.59%
7,Conv_Attn_null_True,77.55% ± 0.68%,84.13% ± 0.61%,99.04% ± 0.12%,96.89% ± 0.21%,92.67% ± 0.96%
8,Conv_Max_null_False,95.33% ± 0.85%,91.14% ± 1.00%,99.26% ± 0.12%,98.69% ± 0.11%,87.89% ± 1.79%
9,Conv_Max_null_True,78.47% ± 1.88%,83.23% ± 2.35%,98.83% ± 0.18%,96.76% ± 0.31%,89.47% ± 3.09%


In [23]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics(data.articles, results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

,expr,precision,f1_score,hamming_loss,auc,accuracy,recall
0,Attn_Attn_null_False,78.22% ± 1.61%,70.91% ± 0.86%,4.52% ± 0.16%,96.30% ± 0.27%,50.64% ± 1.97%,65.74% ± 1.75%
1,Attn_Attn_null_True,65.86% ± 1.61%,70.55% ± 0.98%,5.61% ± 0.07%,96.67% ± 0.26%,41.69% ± 0.36%,77.02% ± 2.12%
2,Attn_Max_null_False,95.82% ± 0.19%,89.96% ± 0.15%,1.37% ± 0.10%,99.26% ± 0.17%,83.98% ± 0.51%,85.84% ± 0.44%
3,Attn_Max_null_True,78.19% ± 1.92%,84.24% ± 1.68%,3.33% ± 0.44%,98.87% ± 0.11%,59.04% ± 4.26%,92.05% ± 1.29%
4,Attn_Mean_null_False,96.51% ± 0.85%,86.24% ± 1.71%,1.78% ± 0.13%,99.26% ± 0.20%,78.35% ± 1.67%,79.26% ± 2.14%
5,Attn_Mean_null_True,76.51% ± 2.72%,82.77% ± 1.59%,3.50% ± 0.08%,98.88% ± 0.12%,57.69% ± 0.72%,91.10% ± 0.74%
6,Conv_Attn_null_False,94.22% ± 0.88%,91.68% ± 1.37%,1.33% ± 0.16%,99.29% ± 0.16%,84.79% ± 1.64%,89.59% ± 1.59%
7,Conv_Attn_null_True,77.55% ± 0.68%,84.13% ± 0.61%,3.11% ± 0.21%,99.04% ± 0.12%,61.72% ± 1.75%,92.67% ± 0.96%
8,Conv_Max_null_False,95.33% ± 0.85%,91.14% ± 1.00%,1.31% ± 0.11%,99.26% ± 0.12%,85.13% ± 0.69%,87.89% ± 1.79%
9,Conv_Max_null_True,78.47% ± 1.88%,83.23% ± 2.35%,3.24% ± 0.31%,98.83% ± 0.18%,60.60% ± 3.02%,89.47% ± 3.09%


In [24]:
hparams = metrics.apply(lambda x : pd.Series(get_experiment_params(x['expr'])), axis = 1)
metrics = hparams.join(metrics, lsuffix="hparams")
metrics['Experiment'] = metrics['expr']
metrics = metrics.drop(columns = ['expr'])
metrics = metrics[[metrics.columns[-1], *metrics.columns[:-1]]]
# metrics.to_csv(os.path.join(dir, "hierarchical_teams_metrics.xlsx"), index = False)
metrics

,Experiment,GNN Variant,Readout Function,Pooling Layer,Undersampling,precision,f1_score,hamming_loss,auc,accuracy,recall
0,Attn_Attn_null_False,Attn,Attn,null,False,78.22% ± 1.61%,70.91% ± 0.86%,4.52% ± 0.16%,96.30% ± 0.27%,50.64% ± 1.97%,65.74% ± 1.75%
1,Attn_Attn_null_True,Attn,Attn,null,True,65.86% ± 1.61%,70.55% ± 0.98%,5.61% ± 0.07%,96.67% ± 0.26%,41.69% ± 0.36%,77.02% ± 2.12%
2,Attn_Max_null_False,Attn,Max,null,False,95.82% ± 0.19%,89.96% ± 0.15%,1.37% ± 0.10%,99.26% ± 0.17%,83.98% ± 0.51%,85.84% ± 0.44%
3,Attn_Max_null_True,Attn,Max,null,True,78.19% ± 1.92%,84.24% ± 1.68%,3.33% ± 0.44%,98.87% ± 0.11%,59.04% ± 4.26%,92.05% ± 1.29%
4,Attn_Mean_null_False,Attn,Mean,null,False,96.51% ± 0.85%,86.24% ± 1.71%,1.78% ± 0.13%,99.26% ± 0.20%,78.35% ± 1.67%,79.26% ± 2.14%
5,Attn_Mean_null_True,Attn,Mean,null,True,76.51% ± 2.72%,82.77% ± 1.59%,3.50% ± 0.08%,98.88% ± 0.12%,57.69% ± 0.72%,91.10% ± 0.74%
6,Conv_Attn_null_False,Conv,Attn,null,False,94.22% ± 0.88%,91.68% ± 1.37%,1.33% ± 0.16%,99.29% ± 0.16%,84.79% ± 1.64%,89.59% ± 1.59%
7,Conv_Attn_null_True,Conv,Attn,null,True,77.55% ± 0.68%,84.13% ± 0.61%,3.11% ± 0.21%,99.04% ± 0.12%,61.72% ± 1.75%,92.67% ± 0.96%
8,Conv_Max_null_False,Conv,Max,null,False,95.33% ± 0.85%,91.14% ± 1.00%,1.31% ± 0.11%,99.26% ± 0.12%,85.13% ± 0.69%,87.89% ± 1.79%
9,Conv_Max_null_True,Conv,Max,null,True,78.47% ± 1.88%,83.23% ± 2.35%,3.24% ± 0.31%,98.83% ± 0.18%,60.60% ± 3.02%,89.47% ± 3.09%


- Visualizations

In [25]:
def bar_plot(df : pd.DataFrame, metric : str) -> None:

    fig = px.bar(
        data_frame = df.sort_values(by = f'{metric}_mean'),
        x = 'expr',
        y = f'{metric}_mean',
        barmode = 'group',
        error_y = f'{metric}_std',
        text = f'{metric}_mean',
        width = 950,
        height = 800
    )

    fig.update_layout(
        xaxis_title = 'Experiment',
        yaxis_title = metric.capitalize().replace('_',' '),
    )

    fig.update_traces(
        textposition = 'outside',
        texttemplate = '%{text:.2%}'
    )

    fig.show()

In [26]:
bar_plot(label_wise_metrics_agg, 'accuracy')

In [27]:
bar_plot(label_wise_metrics_agg, 'f1_score')

In [28]:
bar_plot(row_wise_metrics_agg, 'accuracy')